# IT3051 Fundamentals of Data Mining — Model Comparison & Business Integration

## Project: Soft E-commerce Customer Segmentation and Retention Prioritization

### Purpose and Overview
This notebook consolidates the results from all four clustering algorithms:
1. **K-Means** (Hard clustering baseline)
2. **Gaussian Mixture Model (GMM)** (Probabilistic soft clustering)
3. **Fuzzy C-Means (FCM)** (Fuzzy continuous membership)
4. **Agglomerative Hierarchical Clustering** (Hierarchical baseline)

### Objectives
1. **Model Comparison:** Compare clustering performance across internal validity metrics (Silhouette Score, Davies-Bouldin Index, Calinski-Harabasz).
2. **Soft Clustering Selection:** Compare membership entropy and ambiguity between GMM and FCM to select the best soft-clustering model.
3. **Business Prioritization:** Merge the chosen model's Segment Ambiguity Score with Customer Lifetime Value (`customer_lifetime_value_usd`) and Churn Risk (`churn_risk_score`) from `development_business_data.csv` to identify high-priority retention targets.
4. **Holdout Evaluation:** Validate the final selected pipeline and customer segments on `holdout_preprocessed.csv` and `holdout_business_data.csv`.

### Critical Group Constraints
- All evaluations must use the identical development and holdout splits generated by `01_EDA_Preprocessing.ipynb`.
- No algorithm may use an independent feature subset or alternate data scaling.

---
*Note: Placeholder notebook. Synthesis and comparison will be executed after model training in individual notebooks.*


In [ ]:
# Setup and imports
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Resolve project directories
current_dir = Path.cwd().resolve()
PROJECT_ROOT = current_dir.parent if current_dir.name == "notebooks" else current_dir
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"
MODELS_DIR = PROJECT_ROOT / "models"
FIGURES_DIR = PROJECT_ROOT / "results" / "figures"

print("Project root:", PROJECT_ROOT)
print("Processed data dir:", PROCESSED_DATA_DIR)



# Model Comparison and Final Model Selection

## Objective

This notebook compares the four clustering algorithms developed
independently using the common preprocessing pipeline:

1. K-Means
2. Gaussian Mixture Model (GMM)
3. Fuzzy C-Means (FCM)
4. Agglomerative Hierarchical Clustering

The purpose is to compare the selected/tuned configuration of each
algorithm using common clustering-quality metrics, stability,
generalization evidence, computational practicality, business
interpretability, and alignment with the soft customer segmentation
objective.

The final project model will be selected only after this comparison.

Important:
- K-Means, GMM and FCM were developed using the full 40,000-customer
  development dataset.
- Agglomerative Clustering was developed on a reproducible 2,500-customer
  development sample because of hierarchical clustering's memory
  requirements.
- Therefore, Agglomerative metric values are sample-based and will not
  be treated as directly equivalent to full-development metrics without
  this limitation being stated.

In [1]:
# ============================================================
# Cell 2 — Imports and Project Paths
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
import json
import joblib
import warnings

warnings.filterwarnings("ignore")

# ------------------------------------------------------------
# Project paths
# ------------------------------------------------------------

# Notebook is inside /notebooks, so project root is one level up
PROJECT_ROOT = Path.cwd().parent

DATA_DIR = PROJECT_ROOT / "data" / "processed"
RESULTS_DIR = PROJECT_ROOT / "results"
MODELS_DIR = PROJECT_ROOT / "models"

# ------------------------------------------------------------
# Important shared Phase 1 files
# ------------------------------------------------------------

DEV_PREPROCESSED = DATA_DIR / "development_preprocessed.csv"
HOLDOUT_PREPROCESSED = DATA_DIR / "holdout_preprocessed.csv"

DEV_RAW = DATA_DIR / "development_raw.csv"
HOLDOUT_RAW = DATA_DIR / "holdout_raw.csv"

DEV_BUSINESS = DATA_DIR / "development_business_data.csv"
HOLDOUT_BUSINESS = DATA_DIR / "holdout_business_data.csv"

DEV_PROFILING = DATA_DIR / "development_profiling_data.csv"
HOLDOUT_PROFILING = DATA_DIR / "holdout_profiling_data.csv"

PREPROCESSOR_PATH = MODELS_DIR / "modelling_preprocessor.pkl"

# ------------------------------------------------------------
# Basic verification
# ------------------------------------------------------------

required_files = {
    "Development preprocessed": DEV_PREPROCESSED,
    "Holdout preprocessed": HOLDOUT_PREPROCESSED,
    "Development raw": DEV_RAW,
    "Holdout raw": HOLDOUT_RAW,
    "Development business": DEV_BUSINESS,
    "Holdout business": HOLDOUT_BUSINESS,
    "Development profiling": DEV_PROFILING,
    "Holdout profiling": HOLDOUT_PROFILING,
    "Preprocessor": PREPROCESSOR_PATH,
}

print("PROJECT ROOT:")
print(PROJECT_ROOT)

print("\nSHARED FILE VERIFICATION")
print("=" * 70)

all_available = True

for name, path in required_files.items():
    exists = path.exists()

    if not exists:
        all_available = False

    print(f"{name:<30} {'FOUND' if exists else 'MISSING'}")

print("=" * 70)

if all_available:
    print("All required Phase 1 shared files are available.")
else:
    print("WARNING: One or more required files are missing.")

PROJECT ROOT:
d:\FDM\soft-ecommerce-customer-segmentation

SHARED FILE VERIFICATION
Development preprocessed       FOUND
Holdout preprocessed           FOUND
Development raw                FOUND
Holdout raw                    FOUND
Development business           FOUND
Holdout business               FOUND
Development profiling          FOUND
Holdout profiling              FOUND
Preprocessor                   FOUND
All required Phase 1 shared files are available.


In [2]:
# ============================================================
# Cell 3 — Load and Verify Shared Datasets
# ============================================================

# Load model datasets
dev_preprocessed = pd.read_csv(DEV_PREPROCESSED)
holdout_preprocessed = pd.read_csv(HOLDOUT_PREPROCESSED)

# Load original selected features
dev_raw = pd.read_csv(DEV_RAW)
holdout_raw = pd.read_csv(HOLDOUT_RAW)

# Load business variables
dev_business = pd.read_csv(DEV_BUSINESS)
holdout_business = pd.read_csv(HOLDOUT_BUSINESS)

# Load profiling variables
dev_profiling = pd.read_csv(DEV_PROFILING)
holdout_profiling = pd.read_csv(HOLDOUT_PROFILING)

# Separate customer ID from clustering features
X_dev = dev_preprocessed.drop(columns=["customer_id"])
X_holdout = holdout_preprocessed.drop(columns=["customer_id"])

# ------------------------------------------------------------
# Verification
# ------------------------------------------------------------

print("DATASET VERIFICATION")
print("=" * 65)

print(f"Development customers      : {len(X_dev):,}")
print(f"Holdout customers          : {len(X_holdout):,}")
print(f"Model features             : {X_dev.shape[1]}")

print(f"\nX_dev shape                : {X_dev.shape}")
print(f"X_holdout shape            : {X_holdout.shape}")

print(f"\nMissing values in X_dev    : {X_dev.isna().sum().sum()}")
print(f"Missing values in X_holdout: {X_holdout.isna().sum().sum()}")

print(f"\nInfinite values in X_dev   : {np.isinf(X_dev.to_numpy()).sum()}")
print(f"Infinite values in holdout : {np.isinf(X_holdout.to_numpy()).sum()}")

# Check feature consistency
same_features = list(X_dev.columns) == list(X_holdout.columns)

print(f"\nSame feature structure     : {same_features}")

assert X_dev.shape == (40000, 18), \
    f"Unexpected development shape: {X_dev.shape}"

assert X_holdout.shape == (10000, 18), \
    f"Unexpected holdout shape: {X_holdout.shape}"

assert same_features, \
    "Development and holdout feature columns do not match."

assert X_dev.isna().sum().sum() == 0
assert X_holdout.isna().sum().sum() == 0

assert np.isfinite(X_dev.to_numpy()).all()
assert np.isfinite(X_holdout.to_numpy()).all()

print("\n✓ Shared modelling data verification passed.")

DATASET VERIFICATION
Development customers      : 40,000
Holdout customers          : 10,000
Model features             : 18

X_dev shape                : (40000, 18)
X_holdout shape            : (10000, 18)

Missing values in X_dev    : 0
Missing values in X_holdout: 0

Infinite values in X_dev   : 0
Infinite values in holdout : 0

Same feature structure     : True

✓ Shared modelling data verification passed.


In [3]:
# ============================================================
# Cell 4 — Discover Available Model Result Files
# ============================================================

print("AVAILABLE RESULT FILES")
print("=" * 75)

result_files = sorted([
    p for p in RESULTS_DIR.rglob("*")
    if p.is_file()
])

if result_files:
    for file in result_files:
        print(file.relative_to(PROJECT_ROOT))
else:
    print("No result files found.")

print("\nAVAILABLE MODEL FILES")
print("=" * 75)

model_files = sorted([
    p for p in MODELS_DIR.rglob("*")
    if p.is_file()
])

if model_files:
    for file in model_files:
        print(file.relative_to(PROJECT_ROOT))
else:
    print("No model files found.")

AVAILABLE RESULT FILES
results\agglomerative\agglomerative_baseline_sample_metrics.csv
results\agglomerative\agglomerative_experiment_sample_results.csv
results\agglomerative\agglomerative_final_sample_summary.csv
results\agglomerative\agglomerative_final_sample_summary.json
results\agglomerative\agglomerative_reproducibility.json
results\agglomerative\agglomerative_sample_representativeness.csv
results\agglomerative\agglomerative_selected_sample_assignments.csv
results\agglomerative\agglomerative_selected_sample_cluster_sizes.csv
results\agglomerative\agglomerative_selected_sample_profiles.csv
results\agglomerative\agglomerative_stability_sample_results.csv
results\fcm_cluster_distributions.csv
results\fcm_cluster_profiles.csv
results\fcm_development_holdout_comparison.csv
results\fcm_development_memberships.csv
results\fcm_development_memberships_with_uncertainty.csv
results\fcm_final_robustness.csv
results\fcm_final_summary.csv
results\fcm_holdout_memberships.csv
results\fcm_kmeans_

In [4]:
# ============================================================
# Cell 5 — Inspect Saved Final Model Results
# ============================================================

candidate_files = {
    "K-Means": [
        RESULTS_DIR / "kmeans_final_summary.csv",
        RESULTS_DIR / "kmeans_model_comparison_record.csv",
        RESULTS_DIR / "kmeans" / "kmeans_final_summary.csv",
        RESULTS_DIR / "kmeans" / "kmeans_model_comparison_record.csv",
    ],

    "GMM": [
        RESULTS_DIR / "gmm_final_summary.csv",
        RESULTS_DIR / "gmm_model_comparison_record.csv",
        RESULTS_DIR / "gmm" / "gmm_final_summary.csv",
        RESULTS_DIR / "gmm" / "gmm_model_comparison_record.csv",
    ],

    "FCM": [
        RESULTS_DIR / "fcm_final_summary.csv",
        RESULTS_DIR / "fcm_model_comparison_record.csv",
    ],

    "Agglomerative": [
        RESULTS_DIR / "agglomerative" / "agglomerative_final_sample_summary.csv",
        RESULTS_DIR / "agglomerative" / "agglomerative_final_sample_summary.json",
    ],
}

print("FINAL MODEL RESULT INSPECTION")
print("=" * 80)

for model_name, paths in candidate_files.items():

    print(f"\n{'=' * 25} {model_name} {'=' * 25}")

    found_any = False

    for path in paths:

        if path.exists():

            found_any = True

            print(f"\nFOUND: {path.relative_to(PROJECT_ROOT)}")

            try:

                if path.suffix.lower() == ".csv":

                    df_temp = pd.read_csv(path)

                    print("Shape:", df_temp.shape)
                    print("Columns:")
                    print(df_temp.columns.tolist())

                    print("\nContents:")
                    display(df_temp.head(10))

                elif path.suffix.lower() == ".json":

                    with open(path, "r", encoding="utf-8") as f:
                        json_temp = json.load(f)

                    print("JSON contents:")
                    print(json.dumps(json_temp, indent=2))

            except Exception as e:
                print("Could not read file:", e)

    if not found_any:
        print("\nNo standardized final-summary file found.")

FINAL MODEL RESULT INSPECTION

========================= K-Means =========================

No standardized final-summary file found.

========================= GMM =========================

No standardized final-summary file found.

========================= FCM =========================

FOUND: results\fcm_final_summary.csv
Shape: (1, 21)
Columns:
['algorithm', 'K', 'm', 'initialization', 'development_rows', 'holdout_rows', 'n_features', 'development_silhouette', 'holdout_silhouette', 'development_davies_bouldin', 'holdout_davies_bouldin', 'development_calinski_harabasz', 'holdout_calinski_harabasz', 'development_fpc', 'holdout_fpc', 'development_mean_max_membership', 'holdout_mean_max_membership', 'iterations', 'converged', 'robustness_mean_ARI', 'supports_soft_membership']

Contents:


,algorithm,K,m,initialization,development_rows,holdout_rows,n_features,development_silhouette,holdout_silhouette,development_davies_bouldin,...,development_calinski_harabasz,holdout_calinski_harabasz,development_fpc,holdout_fpc,development_mean_max_membership,holdout_mean_max_membership,iterations,converged,robustness_mean_ARI,supports_soft_membership
0,Fuzzy C-Means,4,1.1,K-Means-informed,40000,10000,18,0.064314,0.061701,2.978749,...,2558.748651,642.931401,0.731309,0.72966,0.812413,0.811196,150,True,1.0,True



FOUND: results\fcm_model_comparison_record.csv
Shape: (1, 16)
Columns:
['algorithm', 'selected_K', 'silhouette_dev', 'silhouette_holdout', 'davies_bouldin_dev', 'davies_bouldin_holdout', 'calinski_harabasz_dev', 'calinski_harabasz_holdout', 'supports_soft_membership', 'soft_measure', 'fpc_dev', 'fpc_holdout', 'converged', 'stability_method', 'stability_ARI', 'business_interpretable']

Contents:


,algorithm,selected_K,silhouette_dev,silhouette_holdout,davies_bouldin_dev,davies_bouldin_holdout,calinski_harabasz_dev,calinski_harabasz_holdout,supports_soft_membership,soft_measure,fpc_dev,fpc_holdout,converged,stability_method,stability_ARI,business_interpretable
0,FCM,4,0.064314,0.061701,2.978749,2.979027,2558.748651,642.931401,True,Fuzzy membership,0.731309,0.72966,True,K-Means-informed perturbation robustness,1.0,True



========================= Agglomerative =========================

FOUND: results\agglomerative\agglomerative_final_sample_summary.csv
Shape: (1, 37)
Columns:
['algorithm', 'author', 'status', 'development_rows', 'holdout_rows_schema_only', 'assignment_rows', 'sample_method', 'sample_seed', 'sample_n', 'silhouette_eval_n', 'unlabeled_development_rows', 'full_development_condensed_distance_gb', 'available_ram_gb_at_start', 'full_fit_working_budget_gb', 'full_fit_memory_safe_under_budget', 'full_development_fit_attempted', 'silhouette_eval_seed', 'selected_K', 'selected_linkage', 'selected_metric', 'selected_actual_clusters', 'selected_silhouette', 'selected_davies_bouldin', 'selected_calinski_harabasz', 'selected_stability_mean_ari', 'selected_stability_min_ari', 'selected_stability_gate_pass', 'selected_fit_runtime_s', 'scipy_cut_vs_sklearn_ari', 'sweep_failure_rows', 'sweep_cluster_below_1pct_rows', 'sample_representativeness_max_abs_smd', 'selection_rule', 'selection_limit', 'scope_

,algorithm,author,status,development_rows,holdout_rows_schema_only,assignment_rows,sample_method,sample_seed,sample_n,silhouette_eval_n,...,selected_fit_runtime_s,scipy_cut_vs_sklearn_ari,sweep_failure_rows,sweep_cluster_below_1pct_rows,sample_representativeness_max_abs_smd,selection_rule,selection_limit,scope_limit,kernel,interpreter
0,AgglomerativeClustering,Dinethya,selected_on_development_sample_only,40000,10000,2500,simple random without replacement,42,2500,2500,...,0.344099,1.0,0,12,0.037877,One best-silhouette scored K per linkage/metri...,Stability gate was added after inspecting firs...,Sample-based Agglomerative result: assignments...,agglomerative-local,C:\Users\User\Desktop\FDM_Ptoj\soft-ecommerce-...



FOUND: results\agglomerative\agglomerative_final_sample_summary.json
JSON contents:
{
  "algorithm": "AgglomerativeClustering",
  "author": "Dinethya",
  "status": "selected_on_development_sample_only",
  "development_rows": 40000,
  "holdout_rows_schema_only": 10000,
  "assignment_rows": 2500,
  "sample_method": "simple random without replacement",
  "sample_seed": 42,
  "sample_n": 2500,
  "silhouette_eval_n": 2500,
  "unlabeled_development_rows": 37500,
  "full_development_condensed_distance_gb": 6.39984,
  "available_ram_gb_at_start": 4.460593152,
  "full_fit_working_budget_gb": 9.59976,
  "full_fit_memory_safe_under_budget": false,
  "full_development_fit_attempted": false,
  "silhouette_eval_seed": null,
  "baseline": {
    "K": 3,
    "linkage": "ward",
    "metric": "euclidean",
    "sample_n": 2500,
    "sample_seed": 42,
    "silhouette_eval_n": 2500,
    "actual_clusters": 3,
    "cluster_counts": "[1229, 375, 896]",
    "cluster_percentages": "[49.16, 15.0, 35.84]",
    "s

In [5]:
# ============================================================
# Cell 6 — Locate K-Means and GMM Saved Outputs
# ============================================================

print("K-MEANS SAVED OUTPUTS")
print("=" * 70)

kmeans_files = sorted([
    p for p in PROJECT_ROOT.rglob("*")
    if p.is_file() and "kmeans" in p.name.lower()
])

for path in kmeans_files:
    print(path.relative_to(PROJECT_ROOT))


print("\n\nGMM SAVED OUTPUTS")
print("=" * 70)

gmm_files = sorted([
    p for p in PROJECT_ROOT.rglob("*")
    if p.is_file() and "gmm" in p.name.lower()
])

for path in gmm_files:
    print(path.relative_to(PROJECT_ROOT))


print("\n\nSUMMARY")
print("=" * 70)
print(f"K-Means related files found : {len(kmeans_files)}")
print(f"GMM related files found     : {len(gmm_files)}")

K-MEANS SAVED OUTPUTS
.git\logs\refs\remotes\origin\KMeans
.git\refs\remotes\origin\KMeans
.venv\Lib\site-packages\sklearn\cluster\__pycache__\_kmeans.cpython-313.pyc
.venv\Lib\site-packages\sklearn\cluster\_kmeans.py
docs\README_KMeans.md
models\kmeans_final_model.pkl
notebooks\02_KMeans.ipynb
results\fcm_kmeans_initialized_results.csv
results\figures\kmeans\kmeans_cluster_centroids_heatmap.png
results\figures\kmeans\kmeans_elbow_curve.png
results\figures\kmeans\kmeans_metrics_vs_k.png
results\figures\kmeans\kmeans_stability_ari.png
results\kmeans\kmeans_baseline_results.csv
results\kmeans\kmeans_dev_assignments.csv
results\kmeans\kmeans_holdout_assignments.csv
results\kmeans\kmeans_stability_results.csv
results\kmeans\kmeans_summary.json
results\kmeans\kmeans_tuning_results.csv


GMM SAVED OUTPUTS
.git\logs\refs\heads\feature\gmm-model
.git\logs\refs\remotes\origin\feature\gmm-model
.git\refs\heads\feature\gmm-model
.git\refs\remotes\origin\feature\gmm-model
models\gmm_final_model.pk

In [6]:
# ============================================================
# Cell 7 — Inspect K-Means Summary and GMM Result Files
# ============================================================

# ------------------------------------------------------------
# 1. K-Means summary
# ------------------------------------------------------------

kmeans_summary_path = RESULTS_DIR / "kmeans" / "kmeans_summary.json"

print("K-MEANS FINAL SUMMARY")
print("=" * 80)

if kmeans_summary_path.exists():

    with open(kmeans_summary_path, "r", encoding="utf-8") as f:
        kmeans_summary = json.load(f)

    print(json.dumps(kmeans_summary, indent=2))

else:
    print("K-Means summary file not found.")


# ------------------------------------------------------------
# 2. Show ONLY useful GMM project files
#    Exclude .git and .venv
# ------------------------------------------------------------

print("\n\nGMM PROJECT FILES")
print("=" * 80)

gmm_project_files = []

for path in PROJECT_ROOT.rglob("*"):

    if not path.is_file():
        continue

    relative = path.relative_to(PROJECT_ROOT)

    # Ignore environment and Git internal files
    if ".venv" in relative.parts or ".git" in relative.parts:
        continue

    if "gmm" in path.name.lower():
        gmm_project_files.append(path)

for path in sorted(gmm_project_files):
    print(path.relative_to(PROJECT_ROOT))

print("\nUseful GMM project files found:", len(gmm_project_files))

K-MEANS FINAL SUMMARY
{
  "algorithm": "K-Means",
  "rationale": "Hard-clustering benchmark for customer segmentation: partitions customers into K disjoint groups by minimizing within-cluster variance over standardized behavioral/engagement features. customer_id, customer_lifetime_value_usd, and churn_risk_score are excluded from the feature matrix (see Section 1).",
  "baseline": {
    "params": {
      "init": "k-means++",
      "n_init": 10,
      "max_iter": 300,
      "random_state": 42
    },
    "k_range": [
      2,
      3,
      4,
      5,
      6,
      7,
      8
    ],
    "metrics_by_k": [
      {
        "K": 2,
        "inertia": 427185.9821602486,
        "silhouette": 0.12033935194435486,
        "davies_bouldin": 2.6792006254286576,
        "calinski_harabasz": 3072.196112680663,
        "n_iter_": 9,
        "runtime_sec": 0.3765130043029785,
        "min_cluster_pct": 16.645,
        "max_cluster_pct": 83.355
      },
      {
        "K": 3,
        "inertia": 401

In [9]:
# ============================================================
# Cell 8 — Load Final Result Sources for All Four Models
# ============================================================

from IPython.display import display as ipy_display

# -------------------------
# K-Means
# -------------------------
kmeans_summary_path = RESULTS_DIR / "kmeans" / "kmeans_summary.json"

with open(kmeans_summary_path, "r", encoding="utf-8") as f:
    kmeans_summary = json.load(f)


# -------------------------
# GMM
# -------------------------
gmm_metrics_path = RESULTS_DIR / "gmm_final_metrics.csv"
gmm_metrics = pd.read_csv(gmm_metrics_path)


# -------------------------
# FCM
# -------------------------
fcm_metrics_path = RESULTS_DIR / "fcm_model_comparison_record.csv"
fcm_metrics = pd.read_csv(fcm_metrics_path)


# -------------------------
# Agglomerative
# -------------------------
agg_metrics_path = (
    RESULTS_DIR
    / "agglomerative"
    / "agglomerative_final_sample_summary.csv"
)

agg_metrics = pd.read_csv(agg_metrics_path)


# ============================================================
# Display results
# ============================================================

print("K-MEANS SUMMARY TOP-LEVEL KEYS")
print("=" * 70)
print(list(kmeans_summary.keys()))


print("\nGMM FINAL METRICS")
print("=" * 70)
print("Columns:")
print(gmm_metrics.columns.tolist())
ipy_display(gmm_metrics)


print("\nFCM FINAL COMPARISON RECORD")
print("=" * 70)
print("Columns:")
print(fcm_metrics.columns.tolist())
ipy_display(fcm_metrics)


print("\nAGGLOMERATIVE FINAL SAMPLE SUMMARY")
print("=" * 70)
print("Columns:")
print(agg_metrics.columns.tolist())
ipy_display(agg_metrics)

K-MEANS SUMMARY TOP-LEVEL KEYS
['algorithm', 'rationale', 'baseline', 'experiments_tested', 'optimization_changes', 'selection_rules', 'best_config', 'common_metrics', 'specific_metric', 'holdout_metrics', 'status', 'is_stable_selection', 'candidates_evaluated', 'artifact_paths']

GMM FINAL METRICS
Columns:
['model', 'n_components', 'covariance_type', 'n_init', 'reg_covar', 'development_silhouette', 'development_davies_bouldin', 'development_calinski_harabasz', 'development_aic', 'development_bic', 'holdout_silhouette', 'holdout_davies_bouldin', 'holdout_calinski_harabasz', 'holdout_log_likelihood', 'holdout_assignment_confidence']


,model,n_components,covariance_type,n_init,reg_covar,development_silhouette,development_davies_bouldin,development_calinski_harabasz,development_aic,development_bic,holdout_silhouette,holdout_davies_bouldin,holdout_calinski_harabasz,holdout_log_likelihood,holdout_assignment_confidence
0,Gaussian Mixture Model,2,tied,10,0.000001,0.125411,2.599606,3056.768093,491503.770352,493291.870377,0.065329,3.72546,704.036729,-6.48669,0.907611



FCM FINAL COMPARISON RECORD
Columns:
['algorithm', 'selected_K', 'silhouette_dev', 'silhouette_holdout', 'davies_bouldin_dev', 'davies_bouldin_holdout', 'calinski_harabasz_dev', 'calinski_harabasz_holdout', 'supports_soft_membership', 'soft_measure', 'fpc_dev', 'fpc_holdout', 'converged', 'stability_method', 'stability_ARI', 'business_interpretable']


,algorithm,selected_K,silhouette_dev,silhouette_holdout,davies_bouldin_dev,davies_bouldin_holdout,calinski_harabasz_dev,calinski_harabasz_holdout,supports_soft_membership,soft_measure,fpc_dev,fpc_holdout,converged,stability_method,stability_ARI,business_interpretable
0,FCM,4,0.064314,0.061701,2.978749,2.979027,2558.748651,642.931401,True,Fuzzy membership,0.731309,0.72966,True,K-Means-informed perturbation robustness,1.0,True



AGGLOMERATIVE FINAL SAMPLE SUMMARY
Columns:
['algorithm', 'author', 'status', 'development_rows', 'holdout_rows_schema_only', 'assignment_rows', 'sample_method', 'sample_seed', 'sample_n', 'silhouette_eval_n', 'unlabeled_development_rows', 'full_development_condensed_distance_gb', 'available_ram_gb_at_start', 'full_fit_working_budget_gb', 'full_fit_memory_safe_under_budget', 'full_development_fit_attempted', 'silhouette_eval_seed', 'selected_K', 'selected_linkage', 'selected_metric', 'selected_actual_clusters', 'selected_silhouette', 'selected_davies_bouldin', 'selected_calinski_harabasz', 'selected_stability_mean_ari', 'selected_stability_min_ari', 'selected_stability_gate_pass', 'selected_fit_runtime_s', 'scipy_cut_vs_sklearn_ari', 'sweep_failure_rows', 'sweep_cluster_below_1pct_rows', 'sample_representativeness_max_abs_smd', 'selection_rule', 'selection_limit', 'scope_limit', 'kernel', 'interpreter']


,algorithm,author,status,development_rows,holdout_rows_schema_only,assignment_rows,sample_method,sample_seed,sample_n,silhouette_eval_n,...,selected_fit_runtime_s,scipy_cut_vs_sklearn_ari,sweep_failure_rows,sweep_cluster_below_1pct_rows,sample_representativeness_max_abs_smd,selection_rule,selection_limit,scope_limit,kernel,interpreter
0,AgglomerativeClustering,Dinethya,selected_on_development_sample_only,40000,10000,2500,simple random without replacement,42,2500,2500,...,0.344099,1.0,0,12,0.037877,One best-silhouette scored K per linkage/metri...,Stability gate was added after inspecting firs...,Sample-based Agglomerative result: assignments...,agglomerative-local,C:\Users\User\Desktop\FDM_Ptoj\soft-ecommerce-...


In [10]:
# ============================================================
# Cell 9 — Inspect Exact Final Result Fields
# ============================================================

def print_record(title, df):
    print("\n" + "=" * 80)
    print(title)
    print("=" * 80)

    row = df.iloc[0]

    for column in df.columns:
        print(f"{column:<40} : {row[column]}")


# GMM
print_record(
    "GMM — FINAL RESULT",
    gmm_metrics
)

# FCM
print_record(
    "FCM — FINAL RESULT",
    fcm_metrics
)

# Agglomerative
print_record(
    "AGGLOMERATIVE — FINAL RESULT",
    agg_metrics
)


# ------------------------------------------------------------
# K-Means: print useful final-selection sections from JSON
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("K-MEANS — FINAL SELECTION INFORMATION")
print("=" * 80)

print("Available top-level sections:")
for key in kmeans_summary.keys():
    print(" -", key)

# Print shorter final/selection-related sections only
for key, value in kmeans_summary.items():

    key_lower = key.lower()

    if any(word in key_lower for word in [
        "best",
        "selected",
        "final",
        "stability",
        "holdout",
        "selection"
    ]):
        print(f"\n--- {key} ---")

        if isinstance(value, (dict, list)):
            print(json.dumps(value, indent=2))
        else:
            print(value)


GMM — FINAL RESULT
model                                    : Gaussian Mixture Model
n_components                             : 2
covariance_type                          : tied
n_init                                   : 10
reg_covar                                : 1e-06
development_silhouette                   : 0.125411288922534
development_davies_bouldin               : 2.5996063674627186
development_calinski_harabasz            : 3056.768093261229
development_aic                          : 491503.7703521182
development_bic                          : 493291.87037660216
holdout_silhouette                       : 0.0653294230328356
holdout_davies_bouldin                   : 3.7254599634437366
holdout_calinski_harabasz                : 704.0367285712437
holdout_log_likelihood                   : -6.486690050784994
holdout_assignment_confidence            : 0.9076108834341536

FCM — FINAL RESULT
algorithm                                : FCM
selected_K                               : 

In [11]:
# ============================================================
# Cell 10 — Standardized Four-Model Comparison Table
# ============================================================

# ---------- Helper ----------
def get_first_existing(mapping, keys, default=np.nan):
    for key in keys:
        if key in mapping:
            return mapping[key]
    return default


# ============================================================
# 1. K-MEANS
# ============================================================

# Find the selected/best K-Means information from the summary
kmeans_selected = {}

for key, value in kmeans_summary.items():
    if isinstance(value, dict):
        key_lower = key.lower()

        if any(word in key_lower for word in ["best", "selected", "final"]):
            kmeans_selected.update(value)

# Also use the saved assignment files to identify selected K
kmeans_dev_assignments = pd.read_csv(
    RESULTS_DIR / "kmeans" / "kmeans_dev_assignments.csv"
)

kmeans_holdout_assignments = pd.read_csv(
    RESULTS_DIR / "kmeans" / "kmeans_holdout_assignments.csv"
)

# Load fitted final model
kmeans_model = joblib.load(
    MODELS_DIR / "kmeans_final_model.pkl"
)

kmeans_k = getattr(kmeans_model, "n_clusters", np.nan)


# ============================================================
# 2. GMM
# ============================================================

gmm = gmm_metrics.iloc[0]


# ============================================================
# 3. FCM
# ============================================================

fcm = fcm_metrics.iloc[0]


# ============================================================
# 4. AGGLOMERATIVE
# ============================================================

agg = agg_metrics.iloc[0]


# ============================================================
# Build standardized comparison table
# ============================================================

comparison = pd.DataFrame([
    {
        "Model": "K-Means",
        "Development_Data": "40,000",
        "Selected_K": kmeans_k,
        "Soft_Clustering": "No",
        "Holdout_Assignment": "Yes",
        "Important_Limitation":
            "Final K-Means selection did not satisfy the defined stability requirement"
    },

    {
        "Model": "GMM",
        "Development_Data": "40,000",
        "Selected_K": int(gmm["n_components"]),
        "Soft_Clustering": "Yes",
        "Holdout_Assignment": "Yes",
        "Important_Limitation":
            "Probabilistic model assumptions and covariance structure affect the solution"
    },

    {
        "Model": "FCM",
        "Development_Data": "40,000",
        "Selected_K": int(fcm["selected_K"]),
        "Soft_Clustering": "Yes",
        "Holdout_Assignment": "Yes",
        "Important_Limitation":
            "Random initialization was unstable; K-Means-informed initialization was required"
    },

    {
        "Model": "Agglomerative",
        "Development_Data": "2,500 sample",
        "Selected_K": np.nan,
        "Soft_Clustering": "No",
        "Holdout_Assignment": "No",
        "Important_Limitation":
            "Selected on a 2,500-customer development sample; no natural unseen-data prediction"
    }
])

print("STANDARDIZED MODEL CHARACTERISTICS")
print("=" * 90)

ipy_display(comparison)

STANDARDIZED MODEL CHARACTERISTICS


,Model,Development_Data,Selected_K,Soft_Clustering,Holdout_Assignment,Important_Limitation
0,K-Means,"40,000",2.0,No,Yes,Final K-Means selection did not satisfy the de...
1,GMM,"40,000",2.0,Yes,Yes,Probabilistic model assumptions and covariance...
2,FCM,"40,000",4.0,Yes,Yes,Random initialization was unstable; K-Means-in...
3,Agglomerative,"2,500 sample",NaN,No,No,"Selected on a 2,500-customer development sampl..."


In [12]:
# ============================================================
# Cell 11 — Extract Common Numerical Metrics
# ============================================================

# ------------------------------------------------------------
# K-MEANS
# Use final fitted model's K and locate matching K in metrics_by_k
# ------------------------------------------------------------

kmeans_k = int(kmeans_model.n_clusters)

kmeans_metrics_by_k = kmeans_summary.get("metrics_by_k", [])

kmeans_final_metrics = None

for record in kmeans_metrics_by_k:
    if int(record.get("k", -1)) == kmeans_k:
        kmeans_final_metrics = record
        break

print("K-Means selected K:", kmeans_k)

if kmeans_final_metrics is not None:
    print("K-Means metric record found.")
    print(kmeans_final_metrics)
else:
    print("WARNING: Matching K-Means metric record not found.")


# ------------------------------------------------------------
# GMM
# ------------------------------------------------------------

gmm = gmm_metrics.iloc[0]


# ------------------------------------------------------------
# FCM
# ------------------------------------------------------------

fcm = fcm_metrics.iloc[0]


# ------------------------------------------------------------
# AGGLOMERATIVE
# Automatically inspect likely K / metric fields
# ------------------------------------------------------------

agg = agg_metrics.iloc[0]

agg_k_candidates = [
    "selected_k",
    "selected_K",
    "n_clusters",
    "k",
    "K"
]

agg_k = np.nan

for col in agg_k_candidates:
    if col in agg_metrics.columns:
        agg_k = agg[col]
        break


# ------------------------------------------------------------
# Print Agglomerative metric-related fields
# ------------------------------------------------------------

print("\nAGGLOMERATIVE RELEVANT FIELDS")
print("=" * 70)

for col in agg_metrics.columns:

    col_lower = col.lower()

    if any(term in col_lower for term in [
        "cluster",
        "silhouette",
        "davies",
        "calinski",
        "stability",
        "ari",
        "sample",
        "runtime",
        "linkage",
        "metric"
    ]):
        print(f"{col:<40}: {agg[col]}")


print("\nDetected Agglomerative K:", agg_k)

K-Means selected K: 2

AGGLOMERATIVE RELEVANT FIELDS
sample_method                           : simple random without replacement
sample_seed                             : 42
sample_n                                : 2500
silhouette_eval_n                       : 2500
silhouette_eval_seed                    : nan
selected_linkage                        : ward
selected_metric                         : euclidean
selected_actual_clusters                : 2
selected_silhouette                     : 0.1242968045901018
selected_davies_bouldin                 : 2.5844023577357045
selected_calinski_harabasz              : 188.35071578266528
selected_stability_mean_ari             : 0.8340248733001693
selected_stability_min_ari              : 0.7819673713802636
selected_stability_gate_pass            : True
selected_fit_runtime_s                  : 0.3440992000396363
scipy_cut_vs_sklearn_ari                : 1.0
sweep_cluster_below_1pct_rows           : 12
sample_representativeness_max_abs_smd  

In [13]:
# ============================================================
# Cell 12 — Extract Final K-Means Evidence
# ============================================================

kmeans_tuning_path = (
    RESULTS_DIR / "kmeans" / "kmeans_tuning_results.csv"
)

kmeans_stability_path = (
    RESULTS_DIR / "kmeans" / "kmeans_stability_results.csv"
)

# Load files
kmeans_tuning = pd.read_csv(kmeans_tuning_path)
kmeans_stability = pd.read_csv(kmeans_stability_path)

print("K-MEANS TUNING COLUMNS")
print("=" * 70)
print(kmeans_tuning.columns.tolist())

print("\nK-MEANS K=2 TUNING RECORD")
print("=" * 70)

# Automatically detect K column
k_col = None

for candidate in ["k", "K", "n_clusters"]:
    if candidate in kmeans_tuning.columns:
        k_col = candidate
        break

if k_col is not None:

    k2_rows = kmeans_tuning[
        pd.to_numeric(
            kmeans_tuning[k_col],
            errors="coerce"
        ) == 2
    ]

    if len(k2_rows) > 0:

        for _, row in k2_rows.iterrows():
            print("\n--- Record ---")
            for col in kmeans_tuning.columns:
                print(f"{col:<35}: {row[col]}")

    else:
        print("No K=2 record found.")

else:
    print("Could not identify K column.")


print("\n\nK-MEANS STABILITY RESULTS")
print("=" * 70)

for col in kmeans_stability.columns:
    print(col)

print("\nRelevant K=2 stability information:")

if k_col is not None and k_col in kmeans_stability.columns:

    stability_k2 = kmeans_stability[
        pd.to_numeric(
            kmeans_stability[k_col],
            errors="coerce"
        ) == 2
    ]

    ipy_display(stability_k2)

else:
    ipy_display(kmeans_stability)

K-MEANS TUNING COLUMNS
['K', 'init', 'n_init', 'max_iter', 'inertia', 'silhouette', 'davies_bouldin', 'calinski_harabasz', 'n_iter_', 'runtime_sec', 'cluster_sizes', 'min_cluster_pct', 'max_cluster_pct']

K-MEANS K=2 TUNING RECORD

--- Record ---
K                                  : 2
init                               : k-means++
n_init                             : 1
max_iter                           : 300
inertia                            : 430009.85827326303
silhouette                         : 0.0644059453145241
davies_bouldin                     : 3.7459830961018623
calinski_harabasz                  : 2789.398791943526
n_iter_                            : 23
runtime_sec                        : 0.0289583206176757
cluster_sizes                      : {"0": 19993, "1": 20007}
min_cluster_pct                    : 49.9825
max_cluster_pct                    : 50.017500000000005

--- Record ---
K                                  : 2
init                               : k-means++
n_i

,K,silhouette_mean,silhouette_std,davies_bouldin_mean,davies_bouldin_std,calinski_harabasz_mean,calinski_harabasz_std,inertia_mean,inertia_std,pairwise_ari_mean,pairwise_ari_min,pairwise_ari_max,n_pairs,subsample_ari_mean,subsample_ari_min
0,2,0.109292,0.02329,2.889899,0.444427,3019.414848,111.30447,427712.776398,1110.891905,0.622077,-0.00002,1.0,45,0.797032,0.000109


In [14]:
# ============================================================
# Cell 13 — Common Numerical Model Comparison
# ============================================================

# ------------------------------------------------------------
# K-Means final K=2 tuning record
# Use the first matching K=2 tuning configuration
# ------------------------------------------------------------

kmeans_k2 = kmeans_tuning[
    pd.to_numeric(kmeans_tuning[k_col], errors="coerce") == 2
].iloc[0]


# ------------------------------------------------------------
# Create common comparison table
# ------------------------------------------------------------

metric_comparison = pd.DataFrame([
    {
        "Model": "K-Means",
        "Evaluation_Rows": 40000,
        "K": 2,
        "Dev_Silhouette": float(kmeans_k2["silhouette"]),
        "Dev_Davies_Bouldin": float(kmeans_k2["davies_bouldin"]),
        "Dev_Calinski_Harabasz": float(kmeans_k2["calinski_harabasz"]),
        "Holdout_Silhouette": np.nan,
        "Holdout_Davies_Bouldin": np.nan,
        "Holdout_Calinski_Harabasz": np.nan,
        "Soft_Output": False,
        "Comparison_Note":
            "Full development data; final selection did not satisfy defined stability requirement."
    },

    {
        "Model": "GMM",
        "Evaluation_Rows": 40000,
        "K": int(gmm["n_components"]),
        "Dev_Silhouette": float(gmm["development_silhouette"]),
        "Dev_Davies_Bouldin": float(gmm["development_davies_bouldin"]),
        "Dev_Calinski_Harabasz": float(gmm["development_calinski_harabasz"]),
        "Holdout_Silhouette": float(gmm["holdout_silhouette"]),
        "Holdout_Davies_Bouldin": float(gmm["holdout_davies_bouldin"]),
        "Holdout_Calinski_Harabasz": float(gmm["holdout_calinski_harabasz"]),
        "Soft_Output": True,
        "Comparison_Note":
            "Full development data; probabilistic soft assignments available."
    },

    {
        "Model": "FCM",
        "Evaluation_Rows": 40000,
        "K": int(fcm["selected_K"]),
        "Dev_Silhouette": float(fcm["silhouette_dev"]),
        "Dev_Davies_Bouldin": float(fcm["davies_bouldin_dev"]),
        "Dev_Calinski_Harabasz": float(fcm["calinski_harabasz_dev"]),
        "Holdout_Silhouette": float(fcm["silhouette_holdout"]),
        "Holdout_Davies_Bouldin": float(fcm["davies_bouldin_holdout"]),
        "Holdout_Calinski_Harabasz": float(fcm["calinski_harabasz_holdout"]),
        "Soft_Output": True,
        "Comparison_Note":
            "Full development data; fuzzy memberships available; K-Means-informed initialization."
    },

    {
        "Model": "Agglomerative",
        "Evaluation_Rows": int(agg["sample_n"]),
        "K": int(agg["selected_actual_clusters"]),
        "Dev_Silhouette": float(agg["selected_silhouette"]),
        "Dev_Davies_Bouldin": float(agg["selected_davies_bouldin"]),
        "Dev_Calinski_Harabasz": float(agg["selected_calinski_harabasz"]),
        "Holdout_Silhouette": np.nan,
        "Holdout_Davies_Bouldin": np.nan,
        "Holdout_Calinski_Harabasz": np.nan,
        "Soft_Output": False,
        "Comparison_Note":
            "Metrics are based on a reproducible 2,500-customer development sample."
    }
])


# ------------------------------------------------------------
# Display rounded version
# ------------------------------------------------------------

display_columns = [
    "Model",
    "Evaluation_Rows",
    "K",
    "Dev_Silhouette",
    "Dev_Davies_Bouldin",
    "Dev_Calinski_Harabasz",
    "Holdout_Silhouette",
    "Holdout_Davies_Bouldin",
    "Holdout_Calinski_Harabasz",
    "Soft_Output"
]

print("COMMON NUMERICAL MODEL COMPARISON")
print("=" * 100)

ipy_display(
    metric_comparison[display_columns].round(4)
)

COMMON NUMERICAL MODEL COMPARISON


,Model,Evaluation_Rows,K,Dev_Silhouette,Dev_Davies_Bouldin,Dev_Calinski_Harabasz,Holdout_Silhouette,Holdout_Davies_Bouldin,Holdout_Calinski_Harabasz,Soft_Output
0,K-Means,40000,2,0.0644,3.7460,2789.3988,NaN,NaN,NaN,False
1,GMM,40000,2,0.1254,2.5996,3056.7681,0.0653,3.7255,704.0367,True
2,FCM,40000,4,0.0643,2.9787,2558.7487,0.0617,2.9790,642.9314,True
3,Agglomerative,2500,2,0.1243,2.5844,188.3507,NaN,NaN,NaN,False


In [15]:
# ============================================================
# Cell 14 — Development vs Holdout Generalization
# ============================================================

generalization = pd.DataFrame([
    {
        "Model": "GMM",
        "Dev_Silhouette": float(gmm["development_silhouette"]),
        "Holdout_Silhouette": float(gmm["holdout_silhouette"]),
        "Silhouette_Change": (
            float(gmm["holdout_silhouette"])
            - float(gmm["development_silhouette"])
        ),

        "Dev_Davies_Bouldin": float(gmm["development_davies_bouldin"]),
        "Holdout_Davies_Bouldin": float(gmm["holdout_davies_bouldin"]),
        "DB_Change": (
            float(gmm["holdout_davies_bouldin"])
            - float(gmm["development_davies_bouldin"])
        )
    },

    {
        "Model": "FCM",
        "Dev_Silhouette": float(fcm["silhouette_dev"]),
        "Holdout_Silhouette": float(fcm["silhouette_holdout"]),
        "Silhouette_Change": (
            float(fcm["silhouette_holdout"])
            - float(fcm["silhouette_dev"])
        ),

        "Dev_Davies_Bouldin": float(fcm["davies_bouldin_dev"]),
        "Holdout_Davies_Bouldin": float(fcm["davies_bouldin_holdout"]),
        "DB_Change": (
            float(fcm["davies_bouldin_holdout"])
            - float(fcm["davies_bouldin_dev"])
        )
    }
])

# Percentage change in silhouette
generalization["Silhouette_Change_%"] = (
    generalization["Silhouette_Change"]
    / generalization["Dev_Silhouette"]
    * 100
)

print("DEVELOPMENT → HOLDOUT GENERALIZATION")
print("=" * 90)

ipy_display(generalization.round(4))

DEVELOPMENT → HOLDOUT GENERALIZATION


,Model,Dev_Silhouette,Holdout_Silhouette,Silhouette_Change,Dev_Davies_Bouldin,Holdout_Davies_Bouldin,DB_Change,Silhouette_Change_%
0,GMM,0.1254,0.0653,-0.0601,2.5996,3.7255,1.1259,-47.9079
1,FCM,0.0643,0.0617,-0.0026,2.9787,2.9790,0.0003,-4.0629


In [16]:
# ============================================================
# Cell 15 — Collect Stability / Robustness Evidence
# ============================================================

print("STABILITY / ROBUSTNESS EVIDENCE")
print("=" * 90)


# ------------------------------------------------------------
# K-MEANS
# ------------------------------------------------------------

print("\nK-MEANS")
print("-" * 70)

print("Stability result columns:")
print(kmeans_stability.columns.tolist())

print("\nK=2 stability record:")

if "K" in kmeans_stability.columns:
    km_stab = kmeans_stability[
        pd.to_numeric(kmeans_stability["K"], errors="coerce") == 2
    ]

    for _, row in km_stab.iterrows():
        for col in kmeans_stability.columns:
            print(f"{col:<40}: {row[col]}")
else:
    print("K column not found.")


# ------------------------------------------------------------
# GMM
# Search available GMM final metric fields for stability /
# convergence / confidence information
# ------------------------------------------------------------

print("\n\nGMM")
print("-" * 70)

for col in gmm_metrics.columns:
    col_lower = col.lower()

    if any(term in col_lower for term in [
        "stability",
        "ari",
        "converg",
        "confidence",
        "iteration",
        "runtime"
    ]):
        print(f"{col:<40}: {gmm[col]}")


# ------------------------------------------------------------
# FCM
# ------------------------------------------------------------

print("\n\nFCM")
print("-" * 70)

fcm_robustness_path = RESULTS_DIR / "fcm_final_robustness.csv"
fcm_stability_path = RESULTS_DIR / "fcm_stability_results.csv"

if fcm_robustness_path.exists():
    fcm_robustness = pd.read_csv(fcm_robustness_path)

    print("Final robustness results:")
    ipy_display(fcm_robustness)

if fcm_stability_path.exists():
    fcm_stability = pd.read_csv(fcm_stability_path)

    print("\nRandom-initialization stability results:")
    ipy_display(fcm_stability)


# ------------------------------------------------------------
# AGGLOMERATIVE
# ------------------------------------------------------------

print("\n\nAGGLOMERATIVE")
print("-" * 70)

print(
    "Mean stability ARI:",
    agg["selected_stability_mean_ari"]
)

print(
    "Minimum stability ARI:",
    agg["selected_stability_min_ari"]
)

print(
    "Stability gate passed:",
    agg["selected_stability_gate_pass"]
)

STABILITY / ROBUSTNESS EVIDENCE

K-MEANS
----------------------------------------------------------------------
Stability result columns:
['K', 'silhouette_mean', 'silhouette_std', 'davies_bouldin_mean', 'davies_bouldin_std', 'calinski_harabasz_mean', 'calinski_harabasz_std', 'inertia_mean', 'inertia_std', 'pairwise_ari_mean', 'pairwise_ari_min', 'pairwise_ari_max', 'n_pairs', 'subsample_ari_mean', 'subsample_ari_min']

K=2 stability record:
K                                       : 2.0
silhouette_mean                         : 0.1092919731117783
silhouette_std                          : 0.0232900296242901
davies_bouldin_mean                     : 2.889898753101995
davies_bouldin_std                      : 0.4444265035365813
calinski_harabasz_mean                  : 3019.414847682522
calinski_harabasz_std                   : 111.3044696797549
inertia_mean                            : 427712.7763976051
inertia_std                             : 1110.8919054433077
pairwise_ari_mean       

,K,m,mean_ARI,min_ARI,std_ARI,mean_FPC,std_FPC,mean_iterations,max_iterations,all_converged
0,3,1.05,1.0,1.0,0.0,0.898506,5.118656e-11,35.0,35,True
1,4,1.10,1.0,1.0,0.0,0.731309,6.013671e-10,150.0,150,True
2,7,1.10,1.0,1.0,0.0,0.676625,2.514757e-10,769.0,769,True



Random-initialization stability results:


,K,m,mean_ARI,min_ARI,max_ARI,std_ARI,mean_FPC,std_FPC
0,3,1.05,0.196290,0.001815,1.000000,0.291883,0.880501,0.023207
1,4,1.10,0.100147,0.007487,0.265565,0.082197,0.664865,0.001928
2,6,1.10,0.132258,0.078583,0.241358,0.062333,0.665037,0.028452
3,7,1.10,0.171575,0.101642,0.248288,0.054338,0.654972,0.025923
4,8,1.20,0.253756,0.145553,0.405521,0.085505,0.219810,0.000457




AGGLOMERATIVE
----------------------------------------------------------------------
Mean stability ARI: 0.8340248733001693
Minimum stability ARI: 0.7819673713802636
Stability gate passed: True


In [17]:
# ============================================================
# Cell 16 — Standardized Stability / Robustness Summary
# ============================================================

stability_comparison = pd.DataFrame([
    {
        "Model": "K-Means",
        "Stability_Evidence":
            "Repeated-run / subsample ARI analysis",
        "Key_Result":
            "Pairwise ARI mean ≈ 0.622; subsample ARI mean ≈ 0.797",
        "Assessment":
            "Variable stability; final selection did not satisfy the defined stability requirement",
        "Important_Note":
            "Some runs were highly similar while others differed substantially"
    },

    {
        "Model": "GMM",
        "Stability_Evidence":
            "Convergence / assignment-confidence evidence",
        "Key_Result":
            f"Holdout assignment confidence ≈ {float(gmm['holdout_assignment_confidence']):.3f}",
        "Assessment":
            "No directly comparable ARI stability experiment available in the saved final metrics",
        "Important_Note":
            "Do not interpret assignment confidence as ARI stability"
    },

    {
        "Model": "FCM",
        "Stability_Evidence":
            "Random-seed ARI + perturbation robustness",
        "Key_Result":
            "Random init unstable; selected K=4,m=1.10 achieved local perturbation ARI = 1.000",
        "Assessment":
            "Sensitive to random initialization, but locally robust after K-Means-informed initialization",
        "Important_Note":
            "ARI=1.0 applies to small perturbations around informed initialization, not arbitrary random starts"
    },

    {
        "Model": "Agglomerative",
        "Stability_Evidence":
            "Sample stability ARI",
        "Key_Result":
            f"Mean ARI = {float(agg['selected_stability_mean_ari']):.3f}; "
            f"minimum ARI = {float(agg['selected_stability_min_ari']):.3f}",
        "Assessment":
            "Passed its defined stability gate on the sampled analysis",
        "Important_Note":
            "Evidence applies to the 2,500-customer sample-based workflow"
    }
])

print("STANDARDIZED STABILITY / ROBUSTNESS COMPARISON")
print("=" * 100)

ipy_display(stability_comparison)

STANDARDIZED STABILITY / ROBUSTNESS COMPARISON


,Model,Stability_Evidence,Key_Result,Assessment,Important_Note
0,K-Means,Repeated-run / subsample ARI analysis,Pairwise ARI mean ≈ 0.622; subsample ARI mean ...,Variable stability; final selection did not sa...,Some runs were highly similar while others dif...
1,GMM,Convergence / assignment-confidence evidence,Holdout assignment confidence ≈ 0.908,No directly comparable ARI stability experimen...,Do not interpret assignment confidence as ARI ...
2,FCM,Random-seed ARI + perturbation robustness,"Random init unstable; selected K=4,m=1.10 achi...","Sensitive to random initialization, but locall...",ARI=1.0 applies to small perturbations around ...
3,Agglomerative,Sample stability ARI,Mean ARI = 0.834; minimum ARI = 0.782,Passed its defined stability gate on the sampl...,"Evidence applies to the 2,500-customer sample-..."


In [18]:
# ============================================================
# Cell 17 — Project Objective and Deployment Suitability
# ============================================================

objective_comparison = pd.DataFrame([
    {
        "Model": "K-Means",
        "Clustering_Type": "Hard",
        "Customer_Output": "Single cluster label",
        "Soft_Membership": "No",
        "Ambiguity_Support": "No direct support",
        "Unseen_Customer_Assignment": "Yes",
        "Full_Development_Data": "Yes",
        "Project_Role":
            "Hard-clustering benchmark"
    },

    {
        "Model": "GMM",
        "Clustering_Type": "Probabilistic / Soft",
        "Customer_Output": "Cluster probabilities",
        "Soft_Membership": "Yes",
        "Ambiguity_Support": "Yes",
        "Unseen_Customer_Assignment": "Yes",
        "Full_Development_Data": "Yes",
        "Project_Role":
            "Probabilistic soft-segmentation candidate"
    },

    {
        "Model": "FCM",
        "Clustering_Type": "Fuzzy / Soft",
        "Customer_Output": "Membership degrees",
        "Soft_Membership": "Yes",
        "Ambiguity_Support": "Yes",
        "Unseen_Customer_Assignment": "Yes",
        "Full_Development_Data": "Yes",
        "Project_Role":
            "Fuzzy soft-segmentation candidate"
    },

    {
        "Model": "Agglomerative",
        "Clustering_Type": "Hard / Hierarchical",
        "Customer_Output": "Single cluster label",
        "Soft_Membership": "No",
        "Ambiguity_Support": "No direct support",
        "Unseen_Customer_Assignment": "No natural predict()",
        "Full_Development_Data": "No — 2,500 sample",
        "Project_Role":
            "Hierarchical structural benchmark"
    }
])

print("PROJECT OBJECTIVE AND DEPLOYMENT COMPARISON")
print("=" * 100)

ipy_display(objective_comparison)

PROJECT OBJECTIVE AND DEPLOYMENT COMPARISON


,Model,Clustering_Type,Customer_Output,Soft_Membership,Ambiguity_Support,Unseen_Customer_Assignment,Full_Development_Data,Project_Role
0,K-Means,Hard,Single cluster label,No,No direct support,Yes,Yes,Hard-clustering benchmark
1,GMM,Probabilistic / Soft,Cluster probabilities,Yes,Yes,Yes,Yes,Probabilistic soft-segmentation candidate
2,FCM,Fuzzy / Soft,Membership degrees,Yes,Yes,Yes,Yes,Fuzzy soft-segmentation candidate
3,Agglomerative,Hard / Hierarchical,Single cluster label,No,No direct support,No natural predict(),"No — 2,500 sample",Hierarchical structural benchmark


## Interpretation of the Model Comparison

The four algorithms provide different strengths and limitations.

### K-Means

K-Means provides a computationally efficient hard-clustering benchmark and can assign new customers to clusters. However, the selected K-Means solution showed variable stability and did not satisfy the stability requirement defined in its individual modelling workflow. In addition, K-Means produces only a single cluster label for each customer and therefore does not directly represent customer segment ambiguity.

### Gaussian Mixture Model (GMM)

GMM produced the strongest development-set hard-label separation among the full-development models, with a development Silhouette score of approximately 0.1254 and Davies-Bouldin index of approximately 2.5996. It also provides probabilistic cluster assignments, making it suitable for soft customer segmentation.

However, its internal clustering metrics deteriorated on the holdout set. Silhouette decreased from approximately 0.1254 to 0.0653, while Davies-Bouldin increased from approximately 2.5996 to 3.7255. Therefore, the selected GMM solution showed stronger development separation but weaker development-to-holdout consistency.

### Fuzzy C-Means (FCM)

The selected FCM configuration used K=4, m=1.10 and K-Means-informed initialization. Its absolute hard-label separation was weaker than GMM, with a development Silhouette score of approximately 0.0643 and Davies-Bouldin index of approximately 2.9787.

However, its holdout behaviour was highly consistent with its development behaviour. Silhouette changed only from approximately 0.0643 to 0.0617, while Davies-Bouldin remained approximately 2.979 on both datasets.

FCM also directly provides fuzzy membership degrees for every customer. These memberships can be used to quantify how strongly or ambiguously a customer belongs to multiple segments. Random initialization was found to be unstable, but K-Means-informed initialization produced a locally robust selected solution under the perturbation analysis.

### Agglomerative Clustering

Agglomerative Clustering produced encouraging sample-level separation and stability, including a Silhouette score of approximately 0.1243 and mean stability ARI of approximately 0.834.

However, these results were obtained from a reproducible sample of 2,500 development customers rather than the full 40,000-customer development dataset. Standard Agglomerative Clustering also does not naturally provide prediction for unseen customers or soft membership values. Therefore, its results are treated primarily as hierarchical structural evidence rather than directly equivalent full-data deployment evidence.

In [19]:
# ============================================================
# Cell 19 — Inspect Soft-Model Business Profiling Outputs
# ============================================================

print("GMM SAVED OUTPUT FILES")
print("=" * 80)

gmm_files_clean = []

for path in RESULTS_DIR.rglob("*"):
    if path.is_file() and "gmm" in path.name.lower():
        gmm_files_clean.append(path)

for path in sorted(gmm_files_clean):
    print(path.relative_to(PROJECT_ROOT))


print("\n\nFCM CLUSTER PROFILE")
print("=" * 80)

fcm_profile_path = RESULTS_DIR / "fcm_cluster_profiles.csv"

if fcm_profile_path.exists():
    fcm_profiles = pd.read_csv(fcm_profile_path)

    print("Shape:", fcm_profiles.shape)
    print("Columns:")
    print(fcm_profiles.columns.tolist())

    ipy_display(fcm_profiles)

else:
    print("FCM cluster profile file not found.")


print("\n\nGMM ASSIGNMENT FILE")
print("=" * 80)

gmm_assignment_path = RESULTS_DIR / "gmm_cluster_assignments.csv"

if gmm_assignment_path.exists():
    gmm_assignments = pd.read_csv(gmm_assignment_path)

    print("Shape:", gmm_assignments.shape)
    print("Columns:")
    print(gmm_assignments.columns.tolist())

    print("\nFirst 5 rows:")
    ipy_display(gmm_assignments.head())

else:
    print("GMM assignment file not found.")

GMM SAVED OUTPUT FILES
results\gmm_cluster_assignments.csv
results\gmm_final_metrics.csv


FCM CLUSTER PROFILE
Shape: (4, 7)
Columns:
['cluster', 'profile_name', 'development_customers', 'development_percent', 'mean_max_membership', 'mean_normalized_entropy', 'interpretation']


,cluster,profile_name,development_customers,development_percent,mean_max_membership,mean_normalized_entropy,interpretation
0,1,Inactive / High-Churn-Risk Customers,5731,14.3275,0.8998,0.2081,Very long inactivity and elevated churn risk
1,2,Low-Purchase High-Conversion Customers,10836,27.0900,0.8043,0.3636,Lower purchase volume but comparatively high c...
2,3,High-Value Active Customers,12471,31.1775,0.7900,0.3841,"High purchase volume, high CLV and recent acti..."
3,4,Low-Engagement Customers,10962,27.4050,0.8003,0.3672,Lower purchase volume and comparatively low co...




GMM ASSIGNMENT FILE
Shape: (40000, 4)
Columns:
['customer_id', 'cluster', 'assignment_confidence', 'segment_name']

First 5 rows:


,customer_id,cluster,assignment_confidence,segment_name
0,CUST-000001,0,0.999998,Active_Lower_Churn
1,CUST-000002,1,1.000000,Inactive_Higher_Churn
2,CUST-000003,0,0.999998,Active_Lower_Churn
3,CUST-000004,0,1.000000,Active_Lower_Churn
4,CUST-000006,0,1.000000,Active_Lower_Churn


In [20]:
# ============================================================
# Cell 20 — GMM Business / Behavioural Cluster Profiling
# ============================================================

# Merge GMM assignments with original selected development features
gmm_profile_data = (
    dev_raw
    .merge(
        gmm_assignments[
            ["customer_id", "cluster", "assignment_confidence", "segment_name"]
        ],
        on="customer_id",
        how="inner"
    )
)

# Add business variables AFTER clustering
gmm_profile_data = (
    gmm_profile_data
    .merge(
        dev_business,
        on="customer_id",
        how="left"
    )
)

print("GMM PROFILE DATA")
print("=" * 80)

print("Rows:", len(gmm_profile_data))
print("Clusters:", sorted(gmm_profile_data["cluster"].unique()))


# ------------------------------------------------------------
# Cluster sizes
# ------------------------------------------------------------

gmm_cluster_sizes = (
    gmm_profile_data
    .groupby(["cluster", "segment_name"])
    .size()
    .reset_index(name="customers")
)

gmm_cluster_sizes["percentage"] = (
    gmm_cluster_sizes["customers"]
    / len(gmm_profile_data)
    * 100
)

print("\nGMM CLUSTER SIZES")
print("=" * 80)

ipy_display(gmm_cluster_sizes.round(4))


# ------------------------------------------------------------
# Behavioural variables used for interpretation
# ------------------------------------------------------------

behaviour_cols = [
    "tenure_months",
    "total_purchases",
    "avg_order_value_usd",
    "days_since_last_purchase",
    "return_count",
    "complaint_count",
    "satisfaction_score",
    "email_open_rate",
    "click_through_rate",
    "conversion_rate"
]

gmm_behaviour_profile = (
    gmm_profile_data
    .groupby(["cluster", "segment_name"])[behaviour_cols]
    .mean()
    .reset_index()
)

print("\nGMM BEHAVIOURAL PROFILE — MEANS")
print("=" * 80)

ipy_display(gmm_behaviour_profile.round(4))


# ------------------------------------------------------------
# Business variables — post-clustering only
# ------------------------------------------------------------

business_cols = [
    col for col in [
        "customer_lifetime_value_usd",
        "churn_risk_score"
    ]
    if col in gmm_profile_data.columns
]

if business_cols:

    gmm_business_profile = (
        gmm_profile_data
        .groupby(["cluster", "segment_name"])[business_cols]
        .agg(["mean", "median"])
    )

    print("\nGMM BUSINESS PROFILE — POST-CLUSTERING")
    print("=" * 80)

    ipy_display(gmm_business_profile.round(4))


# ------------------------------------------------------------
# Assignment confidence
# ------------------------------------------------------------

gmm_confidence_profile = (
    gmm_profile_data
    .groupby(["cluster", "segment_name"])["assignment_confidence"]
    .agg(["mean", "median", "min", "max"])
    .reset_index()
)

print("\nGMM ASSIGNMENT CONFIDENCE")
print("=" * 80)

ipy_display(gmm_confidence_profile.round(4))

GMM PROFILE DATA
Rows: 40000
Clusters: [np.int64(0), np.int64(1)]

GMM CLUSTER SIZES


,cluster,segment_name,customers,percentage
0,0,Active_Lower_Churn,33837,84.5925
1,1,Inactive_Higher_Churn,6163,15.4075



GMM BEHAVIOURAL PROFILE — MEANS


,cluster,segment_name,tenure_months,total_purchases,avg_order_value_usd,days_since_last_purchase,return_count,complaint_count,satisfaction_score,email_open_rate,click_through_rate,conversion_rate
0,0,Active_Lower_Churn,59.6038,99.5993,505.8506,26.8617,9.5158,6.9945,3.0106,0.4992,0.2511,0.1506
1,1,Inactive_Higher_Churn,60.0232,98.9546,500.4209,259.6012,9.5150,7.0021,3.0140,0.5050,0.2522,0.1501



GMM BUSINESS PROFILE — POST-CLUSTERING


customer_lifetime_value_usd            \
                                                     mean    median   
cluster segment_name                                                  
0       Active_Lower_Churn                      57985.555  42484.23   
1       Inactive_Higher_Churn                   56645.343  41066.62   

                              churn_risk_score         
                                          mean median  
cluster segment_name                                   
0       Active_Lower_Churn             23.3102   22.0  
1       Inactive_Higher_Churn          45.8890   45.0


GMM ASSIGNMENT CONFIDENCE


,cluster,segment_name,mean,median,min,max
0,0,Active_Lower_Churn,0.9958,1.0,0.5002,1.0
1,1,Inactive_Higher_Churn,0.9764,1.0,0.5009,1.0


In [21]:
# ============================================================
# Cell 21 — GMM vs FCM Soft-Segmentation Comparison
# ============================================================

soft_model_comparison = pd.DataFrame([
    {
        "Criterion": "Selected number of segments",
        "GMM": "2",
        "FCM": "4"
    },
    {
        "Criterion": "Soft output",
        "GMM": "Cluster probabilities",
        "FCM": "Fuzzy membership degrees"
    },
    {
        "Criterion": "Development Silhouette",
        "GMM": "0.1254",
        "FCM": "0.0643"
    },
    {
        "Criterion": "Holdout Silhouette",
        "GMM": "0.0653",
        "FCM": "0.0617"
    },
    {
        "Criterion": "Development → Holdout consistency",
        "GMM": "Substantial deterioration",
        "FCM": "High consistency"
    },
    {
        "Criterion": "Business granularity",
        "GMM": "2 broad segments",
        "FCM": "4 more differentiated profiles"
    },
    {
        "Criterion": "Ambiguity analysis",
        "GMM": "Possible using probabilities",
        "FCM": "Directly supported by fuzzy memberships"
    },
    {
        "Criterion": "Important robustness issue",
        "GMM": "No directly comparable ARI test saved",
        "FCM": "Random init unstable; informed init locally robust"
    }
])

print("GMM vs FCM — SOFT SEGMENTATION COMPARISON")
print("=" * 100)

ipy_display(soft_model_comparison)

GMM vs FCM — SOFT SEGMENTATION COMPARISON


,Criterion,GMM,FCM
0,Selected number of segments,2,4
1,Soft output,Cluster probabilities,Fuzzy membership degrees
2,Development Silhouette,0.1254,0.0643
3,Holdout Silhouette,0.0653,0.0617
4,Development → Holdout consistency,Substantial deterioration,High consistency
5,Business granularity,2 broad segments,4 more differentiated profiles
6,Ambiguity analysis,Possible using probabilities,Directly supported by fuzzy memberships
7,Important robustness issue,No directly comparable ARI test saved,Random init unstable; informed init locally ro...


In [22]:
# ============================================================
# Cell 22 — Final Model Selection Evidence
# ============================================================

final_selection_evidence = pd.DataFrame([
    {
        "Criterion": "Development separation",
        "K-Means": "Silhouette 0.0644",
        "GMM": "Silhouette 0.1254",
        "FCM": "Silhouette 0.0643",
        "Agglomerative": "Silhouette 0.1243*"
    },
    {
        "Criterion": "Development → holdout consistency",
        "K-Means": "Not available in standardized evidence",
        "GMM": "Substantial deterioration",
        "FCM": "High consistency",
        "Agglomerative": "Not naturally supported"
    },
    {
        "Criterion": "Soft segmentation",
        "K-Means": "No",
        "GMM": "Yes — probabilities",
        "FCM": "Yes — memberships",
        "Agglomerative": "No"
    },
    {
        "Criterion": "Ambiguity representation",
        "K-Means": "No direct support",
        "GMM": "Possible from probabilities",
        "FCM": "Direct fuzzy-membership support",
        "Agglomerative": "No direct support"
    },
    {
        "Criterion": "Business segmentation",
        "K-Means": "Hard benchmark",
        "GMM": "2 broad segments",
        "FCM": "4 differentiated profiles",
        "Agglomerative": "Hierarchical benchmark"
    },
    {
        "Criterion": "Stability / robustness",
        "K-Means": "Defined stability requirement not satisfied",
        "GMM": "No comparable ARI test saved",
        "FCM": "Random init sensitive; informed init locally robust",
        "Agglomerative": "Sample stability gate passed"
    },
    {
        "Criterion": "Full 40,000 development customers",
        "K-Means": "Yes",
        "GMM": "Yes",
        "FCM": "Yes",
        "Agglomerative": "No — 2,500 sample"
    },
    {
        "Criterion": "Unseen-customer assignment",
        "K-Means": "Yes",
        "GMM": "Yes",
        "FCM": "Yes",
        "Agglomerative": "No natural predict()"
    },
    {
        "Criterion": "Main limitation",
        "K-Means": "Hard output + stability issue",
        "GMM": "Holdout metric deterioration",
        "FCM": "Low absolute separation + init sensitivity",
        "Agglomerative": "Sample-based + deployment limitation"
    }
])

print("FINAL MODEL-SELECTION EVIDENCE")
print("=" * 110)

ipy_display(final_selection_evidence)

FINAL MODEL-SELECTION EVIDENCE


,Criterion,K-Means,GMM,FCM,Agglomerative
0,Development separation,Silhouette 0.0644,Silhouette 0.1254,Silhouette 0.0643,Silhouette 0.1243*
1,Development → holdout consistency,Not available in standardized evidence,Substantial deterioration,High consistency,Not naturally supported
2,Soft segmentation,No,Yes — probabilities,Yes — memberships,No
3,Ambiguity representation,No direct support,Possible from probabilities,Direct fuzzy-membership support,No direct support
4,Business segmentation,Hard benchmark,2 broad segments,4 differentiated profiles,Hierarchical benchmark
5,Stability / robustness,Defined stability requirement not satisfied,No comparable ARI test saved,Random init sensitive; informed init locally r...,Sample stability gate passed
6,"Full 40,000 development customers",Yes,Yes,Yes,"No — 2,500 sample"
7,Unseen-customer assignment,Yes,Yes,Yes,No natural predict()
8,Main limitation,Hard output + stability issue,Holdout metric deterioration,Low absolute separation + init sensitivity,Sample-based + deployment limitation


In [23]:
# ============================================================
# Cell 23 — Final Project Model Selection
# ============================================================

final_model_selection = {
    "selected_model": "Fuzzy C-Means (FCM)",
    "selected_K": 4,
    "fuzziness_m": 1.10,
    "initialization": "K-Means-informed",

    "selection_basis": [
        "Directly supports soft customer segmentation through membership degrees",
        "Supports the planned Segment Ambiguity Score",
        "Uses the complete 40,000-customer development dataset",
        "Can assign memberships to unseen holdout customers",
        "Shows strong development-to-holdout consistency",
        "Produces four differentiated and business-interpretable customer profiles",
        "K-Means-informed initialization produced locally robust solutions"
    ],

    "acknowledged_limitations": [
        "FCM does not have the highest development Silhouette score",
        "Absolute hard-label cluster separation is relatively weak",
        "Random initialization produced unstable solutions",
        "The selected solution depends on K-Means-informed initialization",
        "The final choice is project-objective driven and not based on a single clustering metric"
    ],

    "alternative_model_observation":
        "GMM achieved stronger development hard-label separation and also "
        "supports soft assignments, but its Silhouette and Davies-Bouldin "
        "metrics deteriorated substantially on the holdout data and its "
        "selected solution produced two broad customer segments."
}

print("FINAL PROJECT MODEL")
print("=" * 80)
print("Selected model : Fuzzy C-Means (FCM)")
print("K              : 4")
print("m              : 1.10")
print("Initialization : K-Means-informed")

print("\nSelection rationale:")
for reason in final_model_selection["selection_basis"]:
    print("✓", reason)

print("\nLimitations explicitly acknowledged:")
for limitation in final_model_selection["acknowledged_limitations"]:
    print("•", limitation)

FINAL PROJECT MODEL
Selected model : Fuzzy C-Means (FCM)
K              : 4
m              : 1.10
Initialization : K-Means-informed

Selection rationale:
✓ Directly supports soft customer segmentation through membership degrees
✓ Supports the planned Segment Ambiguity Score
✓ Uses the complete 40,000-customer development dataset
✓ Can assign memberships to unseen holdout customers
✓ Shows strong development-to-holdout consistency
✓ Produces four differentiated and business-interpretable customer profiles
✓ K-Means-informed initialization produced locally robust solutions

Limitations explicitly acknowledged:
• FCM does not have the highest development Silhouette score
• Absolute hard-label cluster separation is relatively weak
• Random initialization produced unstable solutions
• The selected solution depends on K-Means-informed initialization
• The final choice is project-objective driven and not based on a single clustering metric


In [24]:
# ============================================================
# Cell 24 — Save Final Comparison and Selection
# ============================================================

comparison_output_path = RESULTS_DIR / "final_model_comparison.csv"
selection_output_path = RESULTS_DIR / "final_model_selection.json"

metric_comparison.to_csv(
    comparison_output_path,
    index=False
)

with open(selection_output_path, "w", encoding="utf-8") as f:
    json.dump(
        final_model_selection,
        f,
        indent=4
    )

print("SAVED:")
print(comparison_output_path.relative_to(PROJECT_ROOT))
print(selection_output_path.relative_to(PROJECT_ROOT))

SAVED:
results\final_model_comparison.csv
results\final_model_selection.json
